# Notebook 05: ATS Scoring Engine

## Objective
Design and implement a five-component ATS readiness score for all 5,000 synthetic candidates.
Produce a benchmark-ready scored dataset and reusable scoring artifacts for downstream notebooks
and the Streamlit application.

## Methodology
1. Load and validate all required input artifacts
2. Precompute population-level scoring parameters (flag variance weights, skill concentration weights)
3. Implement five scoring components independently
4. Aggregate into a composite 100-point ATS score
5. Validate score distributions by domain and experience
6. Export scored dataset and reusable scoring artifacts

## Score Components
| Component            | Max Points | Basis                                      |
|----------------------|------------|--------------------------------------------|
| Experience           | 25         | Years of experience, graduated scale       |
| Skill Coverage       | 30         | Domain-concentration weighted skill score  |
| Education            | 20         | Normalized education tier                  |
| Experience Flags     | 15         | Variance-weighted binary flag sum          |
| Profile Completeness | 10         | Soft skills and text section presence      |
| Total                | 100        |                                            |

## Assumptions
- ATS score is job-agnostic; it measures resume quality against the population, not a specific role
- Education tiers Masters, MBA, and Postgraduate are treated equivalently
- Unknown education receives a partial score, not zero
- Flag weights are derived from within-domain variance to suppress constant flags
- Skill concentration weights are derived from the full population and applied as a fixed prior
- resume_length_words and pre-computed scores from the raw dataset are excluded
- Scoring functions are deterministic and stateless per candidate

## Inputs
- outputs/processed_resumes.csv
- outputs/normalized_candidate_skill_profiles.csv
- outputs/esco_skill_mapping.csv

## Outputs
- outputs/ats_scored_candidates.csv
- outputs/ats_scoring_artifacts.json

In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

OUTPUT_DIR = Path("../outputs")

# loading input artifacts
resumes = pd.read_csv(OUTPUT_DIR / "processed_resumes.csv")
profiles = pd.read_csv(OUTPUT_DIR / "normalized_candidate_skill_profiles.csv")
esco_mapping = pd.read_csv(OUTPUT_DIR / "esco_skill_mapping.csv")

# merging resume data with skill profiles on candidate_id
df = resumes.merge(profiles, on="candidate_id", how="left")

print("Merged dataset shape:", df.shape)
print()

# confirming all candidates have a skill profile
print("Null skill profiles:", df["normalized_skill_profile"].isnull().sum())
print("Null skill counts:", df["normalized_skill_count"].isnull().sum())
print()

# confirming education normalization carried through
print("highest_education value counts:")
print(df["highest_education"].value_counts(dropna=False).to_string())
print()

# confirming experience range
print("years_experience - min/mean/max:",
      df["years_experience"].min(),
      round(df["years_experience"].mean(), 2),
      df["years_experience"].max())
print()

# confirming domain distribution
print("primary_domain distribution:")
print(df["primary_domain"].value_counts().to_string())

Merged dataset shape: (5000, 39)

Null skill profiles: 0
Null skill counts: 0

highest_education value counts:
highest_education
Postgraduate    1216
Masters         1202
MBA             1185
Bachelors       1127
Unknown          270

years_experience - min/mean/max: 1 6.73 20

primary_domain distribution:
primary_domain
IT              2772
Data Science    1002
HR               501
Legal            397
Engineering      189
Management       139


In [2]:
# flag columns from notebook 01
FLAG_COLS = [
    "management_experience_flag",
    "people_management_flag",
    "project_management_experience_flag",
    "agile_scrum_experience_flag",
    "client_facing_experience_flag",
    "delivery_lead_experience_flag",
    "cloud_experience_flag",
    "ml_experience_flag",
    "compliance_experience_flag",
    "enterprise_systems_experience_flag",
    "offshore_onsite_model_experience_flag",
    "multi_vendor_coordination_flag",
    "process_compliance_experience_flag",
    "documentation_heavy_role_flag",
    "mentoring_experience_flag",
    "stakeholder_management_experience_flag"
]

# computing within-domain variance for each flag
# this captures how much a flag actually discriminates within a domain
# flags that are constant within a domain get near-zero weight
domain_flag_variance = (
    df.groupby("primary_domain")[FLAG_COLS]
    .var()
)

print("Within-domain flag variance by domain:")
print(domain_flag_variance.round(4).to_string())
print()

# averaging variance across domains to get a single weight per flag
# this gives a population-level sense of how discriminative each flag is
mean_flag_variance = domain_flag_variance.mean(axis=0)

print("Mean within-domain variance per flag (higher = more discriminative):")
print(mean_flag_variance.round(4).sort_values(ascending=False).to_string())
print()

# normalizing weights to sum to 1 across all flags
# adding a small epsilon to avoid zero weights entirely
# even a near-constant flag should retain a minimal contribution
EPSILON = 0.01
raw_weights = mean_flag_variance + EPSILON
flag_weights = raw_weights / raw_weights.sum()

print("Normalized flag weights (sum to 1.0):")
print(flag_weights.round(4).sort_values(ascending=False).to_string())
print()
print("Weight sum:", round(flag_weights.sum(), 6))

Within-domain flag variance by domain:
                management_experience_flag  people_management_flag  project_management_experience_flag  agile_scrum_experience_flag  client_facing_experience_flag  delivery_lead_experience_flag  cloud_experience_flag  ml_experience_flag  compliance_experience_flag  enterprise_systems_experience_flag  offshore_onsite_model_experience_flag  multi_vendor_coordination_flag  process_compliance_experience_flag  documentation_heavy_role_flag  mentoring_experience_flag  stakeholder_management_experience_flag
primary_domain                                                                                                                                                                                                                                                                                                                                                                                                                                                         

In [3]:
# parsing normalized skill profiles back into lists for population-level analysis
df["skill_list"] = df["normalized_skill_profile"].apply(
    lambda x: x.split("|") if pd.notna(x) and x != "" else []
)

# building a skill-domain frequency table
# counting how many candidates in each domain have each skill
skill_domain_records = []
for _, row in df.iterrows():
    for skill in row["skill_list"]:
        skill_domain_records.append({
            "skill": skill,
            "domain": row["primary_domain"]
        })

skill_domain_df = pd.DataFrame(skill_domain_records)

# total candidates per domain (denominator for frequency)
domain_totals = df["primary_domain"].value_counts().to_dict()

# skill frequency within each domain (raw counts)
skill_domain_counts = (
    skill_domain_df
    .groupby(["skill", "domain"])
    .size()
    .reset_index(name="count")
)

# computing within-domain frequency rate per skill
skill_domain_counts["domain_total"] = skill_domain_counts["domain"].map(domain_totals)
skill_domain_counts["within_domain_freq"] = (
    skill_domain_counts["count"] / skill_domain_counts["domain_total"]
)

# pivoting to skill x domain matrix
skill_freq_matrix = skill_domain_counts.pivot(
    index="skill", columns="domain", values="within_domain_freq"
).fillna(0)

print("Skill-domain frequency matrix shape:", skill_freq_matrix.shape)
print()
print("Sample rows (top skills by overall frequency):")
print(skill_freq_matrix.head(10).round(3).to_string())
print()

# computing domain concentration score per skill
# using max frequency across domains divided by mean frequency
# a skill that peaks strongly in one domain relative to others is domain-concentrated
# a skill that appears uniformly across all domains is generic
overall_mean = skill_freq_matrix.mean(axis=1)
overall_max = skill_freq_matrix.max(axis=1)

# concentration ratio: how much higher is the peak domain vs the average
# flooring mean at a small value to avoid division by zero for rare skills
concentration_ratio = overall_max / overall_mean.clip(lower=0.001)

print("Skill concentration ratios (higher = more domain-specific):")
print(concentration_ratio.sort_values(ascending=False).round(3).to_string())
print()

# normalizing concentration ratios to use as weights
# these will scale the contribution of each skill to the coverage score
# generic skills contribute less; domain-specific skills contribute more
min_ratio = concentration_ratio.min()
max_ratio = concentration_ratio.max()
skill_concentration_weights = (
    (concentration_ratio - min_ratio) / (max_ratio - min_ratio)
).clip(lower=0.1)  # floor at 0.1 so no skill is completely ignored

print("Normalized skill concentration weights (floored at 0.1):")
print(skill_concentration_weights.sort_values(ascending=False).round(3).to_string())

Skill-domain frequency matrix shape: (35, 6)

Sample rows (top skills by overall frequency):
domain                                      Data Science  Engineering     HR     IT  Legal  Management
skill                                                                                                 
aws                                                0.332        0.307  0.331  0.665  0.325       0.381
azure                                              0.362        0.370  0.327  0.337  0.332       0.237
cad                                                0.000        1.000  0.000  0.000  0.000       0.000
contract drafting                                  0.000        0.000  0.000  0.000  1.000       0.000
docker                                             0.000        0.000  0.000  0.505  0.000       0.000
due diligence                                      0.000        0.000  0.000  0.000  1.000       0.000
employee engagement                                0.000        0.000  0.794  0.000

In [4]:
# counting how many domains each skill appears in (with non-zero frequency)
domains_present = (skill_freq_matrix > 0).sum(axis=1)

print("Number of domains each skill appears in:")
print(domains_present.sort_values().to_string())
print()

# two-tier weighting based on domain presence count
# skills in exactly 1 domain: fully domain-specific, weight 1.0
# skills in 2+ domains: weight = 1 / domains_present, floored at 0.1
# this is interpretable and honest about what the synthetic data supports
def compute_concentration_weight(n_domains):
    if n_domains <= 1:
        return 1.0
    return max(1.0 / n_domains, 0.1)

skill_concentration_weights = domains_present.apply(compute_concentration_weight)

print("Revised skill concentration weights:")
print(skill_concentration_weights.sort_values(ascending=False).round(3).to_string())
print()

# sanity check: confirming generic cross-domain skills get low weights
generic_skills = ["aws", "azure", "gcp", "jira", "servicenow", "use online tools to collaborate"]
print("Weights for known generic skills:")
for s in generic_skills:
    if s in skill_concentration_weights.index:
        print(f"  {s}: {skill_concentration_weights[s]:.3f}")
print()

# confirming domain-specific skills get weight 1.0
specific_skills = ["natural language processing", "hr analytics", "legal research",
                   "cad", "docker", "machine learning"]
print("Weights for known domain-specific skills:")
for s in specific_skills:
    if s in skill_concentration_weights.index:
        print(f"  {s}: {skill_concentration_weights[s]:.3f}")

Number of domains each skill appears in:
skill
contract drafting                              1
cad                                            1
due diligence                                  1
docker                                         1
employee engagement                            1
ensure ongoing compliance with regulations     1
hrms                                           1
hr analytics                                   1
legal research                                 1
kubernetes                                     1
ict project management methodologies           1
java (computer programming)                    1
manufacturing                                  1
manage payroll                                 1
machine learning                               1
linux                                          1
recruit employees                              1
risk management                                1
stakeholder management                         1
solidworks            

In [5]:
# education tier mapping
# Masters, MBA, Postgraduate treated equivalently per prior decisions
# Unknown gets a partial score, not zero
EDUCATION_SCORES = {
    "Postgraduate": 20,
    "Masters":      20,
    "MBA":          20,
    "Bachelors":    14,
    "Unknown":       7
}

def score_education(education_value):
    return EDUCATION_SCORES.get(education_value, 7)


# experience scoring using a graduated scale with diminishing returns
# 1 year = 5 points, 10 years = 22 points, 20 years = 25 points
# log scaling reflects that each additional year matters less at senior levels
def score_experience(years):
    if years <= 0:
        return 0
    # log1p gives a smooth curve; scaled so 20 years maps to 25
    raw = np.log1p(years)
    max_raw = np.log1p(20)
    return round((raw / max_raw) * 25, 2)


# skill coverage using domain-concentration weights
# sum of concentration weights for each skill in the candidate profile
# normalized to a 0-30 scale based on the theoretical maximum achievable
# theoretical max: a candidate with 6 domain-specific skills all at weight 1.0
# that gives a raw weighted sum of 6.0
SKILL_SCORE_MAX_RAW = 6.0

def score_skills(skill_profile_str, concentration_weights):
    if not skill_profile_str or pd.isna(skill_profile_str):
        return 0.0
    skills = skill_profile_str.split("|")
    weighted_sum = sum(concentration_weights.get(s, 0.1) for s in skills)
    # count-based floor: minimum score for having 5 skills at generic weight
    floor_raw = 5 * 0.167
    effective_raw = max(weighted_sum, floor_raw)
    return round(min((effective_raw / SKILL_SCORE_MAX_RAW) * 30, 30), 2)


# flag scoring using precomputed variance-based weights
# weighted sum of active flags, normalized to 0-15
def score_flags(row, flag_cols, flag_weights):
    weighted_sum = sum(row[col] * flag_weights[col] for col in flag_cols)
    # maximum possible is sum of all weights (all flags = 1)
    max_possible = flag_weights.sum()
    return round((weighted_sum / max_possible) * 15, 2)


# profile completeness based on raw field presence
# checking soft_skills_raw, experience_summary, project_summary, key_achievements
# each field contributes proportionally; partial credit for short content
def score_completeness(row):
    fields = ["soft_skills_raw", "experience_summary", "project_summary", "key_achievements"]
    score = 0
    per_field = 10 / len(fields)
    for field in fields:
        val = str(row.get(field, "")).strip()
        if len(val) > 50:
            score += per_field
        elif len(val) > 10:
            score += per_field * 0.5
    return round(score, 2)


# quick validation of scoring functions against known edge cases
print("Education score checks:")
for edu, expected in [("Masters", 20), ("MBA", 20), ("Postgraduate", 20),
                       ("Bachelors", 14), ("Unknown", 7)]:
    print(f"  {edu}: {score_education(edu)} (expected {expected})")
print()

print("Experience score checks:")
for yrs in [1, 3, 5, 10, 15, 20]:
    print(f"  {yrs} years: {score_experience(yrs)}")
print()

print("Skill score checks (using concentration weights dict):")
cw = skill_concentration_weights.to_dict()
# 6 domain-specific skills
test_profile_high = "|".join(["natural language processing", "machine learning",
                               "tensorflow", "pandas", "power bi", "python (computer programming)"])
# 5 generic skills only
test_profile_low = "|".join(["aws", "gcp", "azure", "jira", "servicenow"])
print(f"  6 domain-specific skills: {score_skills(test_profile_high, cw)}")
print(f"  5 generic skills only:    {score_skills(test_profile_low, cw)}")

Education score checks:
  Masters: 20 (expected 20)
  MBA: 20 (expected 20)
  Postgraduate: 20 (expected 20)
  Bachelors: 14 (expected 14)
  Unknown: 7 (expected 7)

Experience score checks:
  1 years: 5.69
  3 years: 11.38
  5 years: 14.71
  10 years: 19.69
  15 years: 22.77
  20 years: 25.0

Skill score checks (using concentration weights dict):
  6 domain-specific skills: 27.5
  5 generic skills only:    4.18


## Step 1: Scoring Function Definitions

Five independent scoring functions are defined below. Each function is stateless
and operates on a single candidate record. Population-level parameters (flag weights,
skill concentration weights) are passed in as precomputed arguments.

Key design decisions carried through from the framework:
- Education tiers Masters, MBA, and Postgraduate are treated equivalently at 20 points
- Unknown education scores 7 points, not zero
- Experience uses a log scale to reflect diminishing returns at senior levels
- Skill coverage uses domain-concentration weights; generic cross-domain tools
  contribute less than domain-specific skills
- Flag scoring uses within-domain variance weights to suppress constant flags
- Profile completeness is computed from raw text field presence, not from the
  pre-computed scores in the original dataset which carry no signal

In [6]:
# converting concentration weights to dict for fast lookup
cw = skill_concentration_weights.to_dict()
fw = flag_weights  # already a Series with flag names as index

# applying all five scoring components to every candidate
df["score_education"]    = df["highest_education"].apply(score_education)
df["score_experience"]   = df["years_experience"].apply(score_experience)
df["score_skills"]       = df["normalized_skill_profile"].apply(
                               lambda x: score_skills(x, cw))
df["score_flags"]        = df.apply(
                               lambda row: score_flags(row, FLAG_COLS, fw), axis=1)
df["score_completeness"] = df.apply(score_completeness, axis=1)

# computing composite ATS score
df["ats_score"] = (
    df["score_education"] +
    df["score_experience"] +
    df["score_skills"] +
    df["score_flags"] +
    df["score_completeness"]
).round(2)

# basic distribution check
print("ATS score distribution:")
print(df["ats_score"].describe().round(2).to_string())
print()

print("Component score distributions:")
components = ["score_education", "score_experience", "score_skills",
              "score_flags", "score_completeness"]
for col in components:
    s = df[col]
    print(f"  {col:<25} min:{s.min():>6.2f}  mean:{s.mean():>6.2f}  "
          f"max:{s.max():>6.2f}  std:{s.std():>5.2f}")
print()

print("ATS score distribution by domain:")
print(df.groupby("primary_domain")["ats_score"]
      .agg(["mean", "std", "min", "max"])
      .round(2).to_string())

ATS score distribution:
count    5000.00
mean       69.07
std        10.61
min        36.90
25%        60.17
50%        69.89
75%        78.26
max        89.96

Component score distributions:
  score_education           min:  7.00  mean: 17.95  max: 20.00  std: 3.61
  score_experience          min:  5.69  mean: 15.67  max: 25.00  std: 4.43
  score_skills              min: 11.67  mean: 18.49  max: 21.67  std: 2.79
  score_flags               min:  0.00  mean:  8.21  max: 14.87  std: 5.52
  score_completeness        min:  8.75  mean:  8.75  max:  8.75  std: 0.00

ATS score distribution by domain:
                 mean    std    min    max
primary_domain                            
Data Science    67.08   9.99  38.24  84.65
Engineering     68.28   9.70  43.11  84.53
HR              72.23  11.01  43.17  89.08
IT              68.62  10.48  36.90  89.96
Legal           70.28  10.09  43.24  84.65
Management      78.86  10.82  43.24  89.96


## Step 2: Score Application and Initial Distribution Analysis

Scores applied to all 5,000 candidates. Overall distribution is well-behaved
with mean 69.07 and std 10.61 across the full range of 36.9 to 90.0.

### Known Synthetic Data Artifact
score_completeness is flat at 8.75 for all candidates. All synthetic text fields
exceed the completeness threshold due to templated generation. This component
will produce meaningful variance on real resume inputs. It is retained in the
scoring architecture and must not be reweighted based on this observation.

### Domain Score Gap Requiring Diagnosis
Management scores 78.86 mean versus Data Science at 67.08. This gap requires
component-level investigation before artifacts are exported.

In [7]:
# breaking down mean component scores by domain to identify the source
component_by_domain = df.groupby("primary_domain")[
    ["score_education", "score_experience", "score_skills",
     "score_flags", "score_completeness", "ats_score"]
].mean().round(2)

print("Mean component scores by domain:")
print(component_by_domain.to_string())
print()

# checking experience distribution by domain
# management has only 139 records - checking if it skews senior
print("Years of experience by domain:")
print(df.groupby("primary_domain")["years_experience"]
      .agg(["mean", "min", "max", "count"])
      .round(2).to_string())
print()

# checking flag score by domain specifically
# management may have higher flag activation rates on the discriminative flags
print("Mean flag score by domain:")
print(df.groupby("primary_domain")["score_flags"].mean().round(2).to_string())
print()

# checking which flags are driving management scores
# comparing management mean flag rates vs population mean
mgmt_flags = df[df["primary_domain"] == "Management"][FLAG_COLS].mean()
pop_flags   = df[FLAG_COLS].mean()
flag_comparison = pd.DataFrame({
    "management_mean": mgmt_flags,
    "population_mean": pop_flags,
    "weight":          flag_weights
}).round(3)
flag_comparison["weighted_gap"] = (
    (flag_comparison["management_mean"] - flag_comparison["population_mean"])
    * flag_comparison["weight"]
).round(4)

print("Flag activation: Management vs population (sorted by weighted gap):")
print(flag_comparison.sort_values("weighted_gap", ascending=False).to_string())

Mean component scores by domain:
                score_education  score_experience  score_skills  score_flags  score_completeness  ats_score
primary_domain                                                                                             
Data Science              17.89             14.61         18.83         7.01                8.75      67.08
Engineering               18.02             13.75         21.67         6.10                8.75      68.28
HR                        17.67             15.87         21.67         8.27                8.75      72.23
IT                        17.99             16.13         16.97         8.78                8.75      68.62
Legal                     18.08             14.66         21.67         7.12                8.75      70.28
Management                17.96             19.13         21.67        11.35                8.75      78.86

Years of experience by domain:
                 mean  min  max  count
primary_domain                  

## Step 3: Domain Score Gap Diagnosis

The 11.8-point gap between Management (78.86) and Data Science (67.08) has three causes:

**Experience gap (primary driver):** Management mean experience is 10.92 years versus
population mean of 6.73. This is a genuine characteristic of the synthetic data and
reflects realistic seniority requirements for management roles. The experience component
correctly rewards this and no adjustment is needed.

**Flag score gap (secondary driver):** Management candidates have higher activation rates
on legitimately discriminative flags including management_experience_flag,
delivery_lead_experience_flag, and multi_vendor_coordination_flag. The variance-based
weighting is functioning as designed.

**Skill score gap (synthetic):** IT and Data Science score lower on skills because
their profiles contain cross-domain tools (aws, gcp, jira) with low concentration weights.
Management, Engineering, HR, and Legal candidates have predominantly single-domain skills
that receive full weight. This will behave differently on real resume data where skill
profiles are more varied.

No scoring adjustments are made. The gap is data-driven and interpretable.
The thin Management population (139 records) means percentile benchmarks for this
domain will carry low confidence, consistent with warnings documented in Memory V2.

In [8]:
# validating score behavior across experience bands
# scores should increase monotonically with experience on average
bins   = [0, 3, 6, 10, 20]
labels = ["Junior (1-3)", "Mid (4-6)", "Senior (7-10)", "Expert (11-20)"]
df["experience_band"] = pd.cut(df["years_experience"], bins=bins, labels=labels)

print("ATS score by experience band:")
print(df.groupby("experience_band", observed=True)["ats_score"]
      .agg(["mean", "std", "min", "max", "count"])
      .round(2).to_string())
print()

# confirming score is monotonically increasing with experience on average
band_means = df.groupby("experience_band", observed=True)["ats_score"].mean()
is_monotonic = all(band_means.iloc[i] < band_means.iloc[i+1]
                   for i in range(len(band_means)-1))
print(f"Score increases monotonically with experience band: {is_monotonic}")
print()

# checking score distribution for Unknown education candidates
# must not be penalized to near-zero
unknown_scores = df[df["highest_education"] == "Unknown"]["ats_score"]
known_scores   = df[df["highest_education"] != "Unknown"]["ats_score"]
print(f"Unknown education - mean ATS: {unknown_scores.mean():.2f}, "
      f"min: {unknown_scores.min():.2f}")
print(f"Known education   - mean ATS: {known_scores.mean():.2f}, "
      f"min: {known_scores.min():.2f}")
print()

# confirming no null or out-of-range scores
print(f"Null ATS scores:          {df['ats_score'].isnull().sum()}")
print(f"Scores below 0:           {(df['ats_score'] < 0).sum()}")
print(f"Scores above 100:         {(df['ats_score'] > 100).sum()}")
print(f"Total candidates scored:  {len(df)}")

ATS score by experience band:
                  mean   std    min    max  count
experience_band                                  
Junior (1-3)     55.64  5.08  36.90  62.67   1171
Mid (4-6)        65.66  5.83  46.78  75.77   1475
Senior (7-10)    76.64  4.98  56.54  84.65   1821
Expert (11-20)   82.20  4.85  66.68  89.96    533

Score increases monotonically with experience band: True

Unknown education - mean ATS: 58.50, min: 36.90
Known education   - mean ATS: 69.68, min: 41.40

Null ATS scores:          0
Scores below 0:           0
Scores above 100:         0
Total candidates scored:  5000


## Step 4: Score Validation

Validation confirms the scoring system behaves correctly across all intended dimensions:

- ATS scores increase monotonically across Junior, Mid, Senior, and Expert experience bands
- Unknown education candidates score 58.50 mean versus 69.68 for known education,
  consistent with the partial score design
- No null scores, no out-of-range values across all 5,000 candidates
- Within-band standard deviation is artificially tight (4-6 points) due to synthetic
  data uniformity; real resume inputs will produce wider within-band distributions

The scoring system is ready for artifact export.

In [10]:
# defining export columns for the scored dataset
score_cols = [
    "candidate_id", "primary_domain", "primary_role", "years_experience",
    "highest_education", "institution_tier", "experience_band",
    "score_education", "score_experience", "score_skills",
    "score_flags", "score_completeness", "ats_score"
]

scored_export = df[score_cols].copy()
scored_export.to_csv(OUTPUT_DIR / "ats_scored_candidates.csv", index=False)
print("ats_scored_candidates.csv exported:", scored_export.shape)
print()

# building scoring artifacts dictionary for reuse in Streamlit application
# this captures everything needed to score a new candidate without recomputing
# population-level parameters

scoring_artifacts = {
    "education_scores": EDUCATION_SCORES,
    "experience_max_years": 20,
    "skill_score_max_raw": SKILL_SCORE_MAX_RAW,
    "skill_score_floor_raw": 5 * 0.167,
    "flag_cols": FLAG_COLS,
    "flag_weights": flag_weights.round(6).to_dict(),
    "skill_concentration_weights": skill_concentration_weights.round(6).to_dict(),
    "completeness_fields": [
        "soft_skills_raw", "experience_summary",
        "project_summary", "key_achievements"
    ],
    "completeness_threshold_full": 50,
    "completeness_threshold_partial": 10,
    "component_max_points": {
        "education":    20,
        "experience":   25,
        "skills":       30,
        "flags":        15,
        "completeness": 10
    },
    "scoring_notes": {
        "education_unknown": "Unknown education receives 7 points, not zero",
        "postgraduate_equivalence": "Masters, MBA, Postgraduate treated equivalently at 20 points",
        "flag_weighting": "Flag weights derived from within-domain variance; constant flags suppressed",
        "skill_weighting": "Skill weights derived from domain concentration; generic tools penalized",
        "completeness_synthetic": "Completeness component flat on synthetic data; meaningful on real resumes"
    }
}

with open(OUTPUT_DIR / "ats_scoring_artifacts.json", "w") as f:
    json.dump(scoring_artifacts, f, indent=2)
print("ats_scoring_artifacts.json exported")
print()

# final validation of exported artifacts
reloaded_scores = pd.read_csv(OUTPUT_DIR / "ats_scored_candidates.csv")
with open(OUTPUT_DIR / "ats_scoring_artifacts.json") as f:
    reloaded_artifacts = json.load(f)

print("Reloaded scored dataset shape:", reloaded_scores.shape)
print("Reloaded artifact keys:", list(reloaded_artifacts.keys()))
print()

# confirming all output files present
print("outputs/ directory contents:")
for f in sorted(OUTPUT_DIR.glob("*.csv")):
    size_kb = round(f.stat().st_size / 1024, 1)
    print(f"  {f.name:<55} {size_kb} KB")
for f in sorted(OUTPUT_DIR.glob("*.json")):
    size_kb = round(f.stat().st_size / 1024, 1)
    print(f"  {f.name:<55} {size_kb} KB")

ats_scored_candidates.csv exported: (5000, 13)

ats_scoring_artifacts.json exported

Reloaded scored dataset shape: (5000, 13)
Reloaded artifact keys: ['education_scores', 'experience_max_years', 'skill_score_max_raw', 'skill_score_floor_raw', 'flag_cols', 'flag_weights', 'skill_concentration_weights', 'completeness_fields', 'completeness_threshold_full', 'completeness_threshold_partial', 'component_max_points', 'scoring_notes']

outputs/ directory contents:
  ats_scored_candidates.csv                               479.4 KB
  candidate_skill_profiles.csv                            292.4 KB
  cleaned_skill_vocabulary.csv                            4.8 KB
  curated_job_descriptions.csv                            2381.1 KB
  domain_distribution_summary.csv                         0.3 KB
  esco_skill_mapping.csv                                  2.7 KB
  job_title_taxonomy.csv                                  15.6 KB
  normalized_candidate_skill_profiles.csv                 724.3 KB
  proce

## Notebook 05 Summary

### Objective Completed
Designed and implemented a five-component ATS readiness score for all 5,000 synthetic
candidates. Produced a benchmark-ready scored dataset and a reusable scoring artifacts
file for downstream notebooks and the Streamlit application.

### Score Components and Results

| Component            | Max Points | Method                              | Mean  | Std  |
|----------------------|------------|-------------------------------------|-------|------|
| Experience           | 25         | Log scale, diminishing returns      | 15.67 | 4.43 |
| Skill Coverage       | 30         | Domain-concentration weighted sum   | 18.49 | 2.79 |
| Education            | 20         | Tier mapping, equivalence enforced  | 17.95 | 3.61 |
| Experience Flags     | 15         | Within-domain variance weighted     | 8.21  | 5.52 |
| Profile Completeness | 10         | Text field presence and length      | 8.75  | 0.00 |
| Total ATS Score      | 100        |                                     | 69.07 | 10.61|

### Decisions Made

Education tiers Masters, MBA, and Postgraduate confirmed equivalent at 20 points.
Unknown education scores 7 points, not zero. resume_length_words and pre-computed
dataset scores excluded throughout.

Skill coverage uses domain-concentration weighting rather than raw count. Weights
derived from number of domains each skill appears in. Single-domain skills receive
weight 1.0. Six-domain tools receive weight 0.167. Python and SQL receive 0.500.
A count-based floor prevents generic skill profiles from scoring near zero.

Flag weights derived from within-domain variance. Four domain-constant flags
(ml_experience_flag, agile_scrum_experience_flag, compliance_experience_flag,
documentation_heavy_role_flag) received only epsilon-derived weights of 0.004.
Behavioral flags (mentoring, enterprise_systems, multi_vendor_coordination)
dominate at approximately 0.10 each.

The 11.8-point Management domain gap was diagnosed and confirmed as data-driven.
Management candidates are genuinely more senior (mean 10.92 years) and more
flag-active than other domains. No scoring adjustment was made.

### Known Limitations

score_completeness is flat at 8.75 for all synthetic candidates due to templated
text fields. This component is retained and will discriminate meaningfully on real
resume inputs. It must not be reweighted based on synthetic data behavior.

Within-band score variance is artificially tight at 4 to 6 points due to synthetic
data uniformity. Real resume inputs will produce wider distributions.

Thin Management (139 records) and Engineering (189 records) populations will produce
low-confidence percentile benchmarks. Deferred to Notebook 07.

### Deferred Decisions
- Thin population handling for Management and Engineering benchmarking
- Percentile binning strategy for the 11-20 year experience tail
- Both deferred to Notebook 07: Benchmarking and Candidate Ranking

### Produced
- outputs/ats_scored_candidates.csv (5000 rows, 13 columns)
- outputs/ats_scoring_artifacts.json (all population-level scoring parameters)